# Efektywne programowanie w języku Python
## Wykład 5 — równoległość i biblioteka standardowa

## Dwa niezależne bloki
- **A. Równoległość:** wybór narzędzia, procesy, wyniki i koszty.
- **B. Biblioteka standardowa:** kontenery, tekst i losowanie.

Przykłady podstawowe: Python 3.10 i NumPy ze środowiska kursu.
Slajdy „Rozszerzenie” są opcjonalne; bloki można prowadzić osobno.

## A. Od problemu do narzędzia
Najpierw poprawny wynik, potem pomiar, dopiero później optymalizacja.

`part04`: potoki i iteratory. Tutaj pytamy, **gdzie** i **jakim kosztem** wykonujemy pracę.

### Współbieżność a równoległość
- **Współbieżność:** obsługa kilku zadań w nakładających się przedziałach czasu.
- **Równoległość:** wykonywanie pracy jednocześnie, np. na kilku rdzeniach.
- **Asynchroniczne API:** wywołanie pozwala kontynuować pracę przed zakończeniem zadania.

Żadne z tych pojęć nie oznacza automatycznie krótszego czasu programu.

### Jak wybrać narzędzie?
| Rodzaj pracy | Punkt wyjścia |
|---|---|
| Małe zadanie | Kod sekwencyjny |
| Operacje na tablicach | Wektoryzacja NumPy |
| Oczekiwanie na I/O | Wątki lub `asyncio` |
| Duże obliczenia w Pythonie | Procesy |

Decyzję potwierdzamy pomiarem na rzeczywistych danych.

### GIL — z zaznaczeniem granic
W standardowym CPythonie 3.10 GIL ogranicza jednoczesne wykonywanie kodu Pythona w wątkach jednego interpretera.

Procesy mają oddzielne interpretery. Operacje I/O i część kodu bibliotek natywnych mogą zwalniać GIL.
Dlatego „wątki nigdy nie przyspieszają obliczeń” jest zbyt szerokim stwierdzeniem.

In [ ]:
import os
import sys
import numpy as np
from pathlib import Path
from time import perf_counter
from statistics import median

print("Python:", sys.version.split()[0], "NumPy:", np.__version__)
print("Logiczne CPU:", os.cpu_count())

### Ile procesów uruchomić?
`os.cpu_count()` i `multiprocessing.cpu_count()` raportują liczbę logicznych CPU, a nie limit procesów.

Dostępność CPU może ograniczać system, kontener lub inne programy. Więcej procesów może zwiększyć narzut i zużycie pamięci.
W demonstracjach używamy najwyżej dwóch procesów.

### Wspólny problem
Dla każdego wiersza policz wartości z przedziału **[4, 8]**.

Granice są włączone. Sprawdzimy tę samą odpowiedź dla pętli, NumPy i procesów.

In [ ]:
rng = np.random.default_rng(100)
arr = rng.integers(0, 10, size=(20_000, 5))
data = arr.tolist()
print(arr.shape)
print(data[:3])

### Ziarno należy do generatora
Używamy **tego samego obiektu** `rng` do tworzenia danych.
Samo wywołanie `np.random.RandomState(100)` bez użycia wyniku nie ustawia ziarna innych generatorów.

Reprodukowalny eksperyment wymaga też zapisania wersji bibliotek i warunków pomiaru.

In [ ]:
from part05_files.workers import count_row

assert count_row([3, 4, 8, 9]) == 2
assert count_row([]) == 0
assert count_row([-5, 0, 10]) == 0
sequential = [count_row(row) for row in data]
print(sequential[:10])

### Czy można zmienić sposób obliczeń?
NumPy wykonuje porównania na całej tablicy; `axis=1` oznacza redukcję w obrębie każdego wiersza.

To wektoryzacja. Sama składnia nie dowodzi, że obliczenia używają wielu rdzeni.

In [ ]:
vectorized = ((arr >= 4) & (arr <= 8)).sum(axis=1)
assert vectorized.tolist() == sequential
print(vectorized[:10])

### Pomiar bez zgubionego wyniku
Wynik obliczamy i sprawdzamy osobno. Kod wewnątrz `%%timeit` nie służy do tworzenia zmiennych dla kolejnych komórek.

Do krótkiego pokazu użyjemy mediany trzech prób. Nie dołączamy wyświetlania wyników ani tworzenia danych.

In [ ]:
def measure(function, expected, repeats=3):
    samples = []
    for _ in range(repeats):
        start = perf_counter()
        result = function()
        samples.append(perf_counter() - start)
        assert result == expected
    return median(samples)

python_seconds = measure(lambda: [count_row(r) for r in data], sequential)
numpy_seconds = measure(
    lambda: ((arr >= 4) & (arr <= 8)).sum(axis=1).tolist(), sequential)
print(f"Python: {python_seconds:.6f} s; NumPy: {numpy_seconds:.6f} s")

### Procesy na Windows i w notebooku
Metoda `spawn` uruchamia nowy interpreter, który importuje potrzebne funkcje.

- Funkcje pracowników: importowalny plik `.py`, bez lambd i funkcji lokalnych.
- Tworzenie procesów w skrypcie: pod `if __name__ == "__main__":`.
- Sam warunek w komórce notebooka nie zapewnia importowalności jej funkcji.

### Pliki do demonstracji
- [workers.py](part05_files/workers.py) — funkcje obliczeniowe.
- [parallel_demo.py](part05_files/parallel_demo.py) — pule, futures i pomiary.

Notebook uruchamia skrypt nowym interpreterem przez `sys.executable`. Import skryptu nie tworzy procesów. Jawne `spawn` daje wspólny punkt odniesienia dla Windows i POSIX.

### `Pool`: blokowanie i kolejność
| Metoda | Co otrzymuje wywołujący? |
|---|---|
| `apply()` | Jeden wynik; czeka na zakończenie |
| `map()`, `starmap()` | Listę wyników w kolejności wejścia |
| `apply_async()` | Uchwyt `AsyncResult` dla jednego zadania |
| `map_async()`, `starmap_async()` | Uchwyt do uporządkowanej listy |

`get()` czeka na wynik i przekazuje wyjątek z zadania.

### Pułapka: blokujące `apply()` w pętli
```python
with context.Pool(2) as pool:
    results = [pool.apply(count_row, (row, 4, 8))
               for row in rows]
```
Każde wywołanie kończy się przed wysłaniem następnego.
Pula istnieje, ale nie korzystamy z niej do równoległego wykonywania tych zadań.

### `map()` i argumenty funkcji
```python
worker = partial(count_row, minimum=4, maximum=8)
with context.Pool(2) as pool:
    results = pool.map(worker, rows)
```
`Pool.map()` przyjmuje jedno iterable. `partial` wiąże dodatkowe argumenty; nie trzeba kopiować funkcji ani tworzyć zbędnej kopii `rows`.

### `starmap()` rozpakowuje argumenty
```python
arguments = [(row, 4, 8) for row in rows]
with context.Pool(2) as pool:
    results = pool.starmap(count_row, arguments)
```
Każda krotka zawiera argumenty jednego wywołania.
Kolejność wyników odpowiada kolejności wejścia.

### `apply_async()`: zgłoś zadania, odbierz wyniki
```python
with context.Pool(2) as pool:
    pending = [pool.apply_async(count_row, (row, 4, 8))
               for row in rows]
    results = [r.get(timeout=20) for r in pending]
```
Funkcja zwraca `int`, więc używamy `r.get()`, bez `[1]`.
Ta pętla odbiera wyniki w kolejności zgłoszeń.

### Zarządzanie pulą
`with Pool(...)` sprząta zasoby również po wyjątku.
W przykładach **odbieramy wszystkie wyniki wewnątrz `with`**.

W ręcznym zarządzaniu `close()` blokuje nowe zadania, a `join()` czeka na zakończenie pracowników. Samo `close()` nie oznacza oczekiwania.

### Uruchomienie rzeczywistych procesów
Skrypt porównuje wszystkie sześć wariantów `Pool` z odpowiedzią sekwencyjną oraz sprawdza `ProcessPoolExecutor`.

Dane są małe, a wszystkie wyniki są odbierane. Nie generujemy setek tysięcy obiektów `AsyncResult`.

In [ ]:
import subprocess

script = Path("part05_files/parallel_demo.py").resolve()
assert script.is_file()
completed = subprocess.run(
    [sys.executable, str(script)], check=True,
    capture_output=True, text=True, encoding="utf-8",
    env={**os.environ, "PYTHONIOENCODING": "utf-8"}, timeout=90)
print(completed.stdout)

### `concurrent.futures`: wspólny interfejs
- `ThreadPoolExecutor`: wątki w jednym procesie.
- `ProcessPoolExecutor`: oddzielne procesy; wymagania importowania i serializacji nadal obowiązują.
- `submit()` zwraca `Future`, `future.result()` odbiera wynik albo wyjątek.
- `executor.map()` zwraca iterator wyników w kolejności wejścia.

Nie zakładamy, że API samo dobierze najlepszą metodę.

### Wątki: symulacja oczekiwania na I/O
`sleep()` modeluje czas oczekiwania, bez zależności od sieci.
Przygotujemy dwa zadania; blokada zdarzenia pozwoli jawnie pokazać kolejność odbioru.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Event
from time import sleep

def delayed_value(label, seconds):
    sleep(seconds)
    return label

with ThreadPoolExecutor(max_workers=2) as executor:
    ordered = list(executor.map(delayed_value, ["A", "B"], [0.02, 0]))
assert ordered == ["A", "B"]
print("map:", ordered)

In [ ]:
release = Event()

def wait_for_release():
    if not release.wait(timeout=5):
        raise TimeoutError("Nie zwolniono zadania A")
    return "A"

with ThreadPoolExecutor(max_workers=2) as executor:
    slow = executor.submit(wait_for_release)
    fast = executor.submit(delayed_value, "B", 0)
    try:
        first = next(as_completed([slow, fast], timeout=3)).result()
    finally:
        release.set()
    second = slow.result(timeout=3)
assert [first, second] == ["B", "A"]
print("as_completed:", [first, second])

### `as_completed()` i wyjątki
`as_completed()` udostępnia gotowe futures, bez zachowania kolejności zgłoszeń. Dla kilku już gotowych zadań nie zakładamy konkretnej kolejności.

Wyjątek z zadania ujawnia się przy odbiorze wyniku. Samo `submit()` nie oznacza sukcesu obliczenia.

In [ ]:
def reciprocal(value):
    return 1 / value

with ThreadPoolExecutor(max_workers=1) as executor:
    future = executor.submit(reciprocal, 0)
    try:
        future.result(timeout=3)
    except ZeroDivisionError:
        print("Odebrano ZeroDivisionError z zadania")
    else:
        raise AssertionError("Brak oczekiwanego wyjątku")

### Rozszerzenie: kto widzi zmianę listy?
Przewidź wynik: zadanie wykonuje `items.append(value)` i zwraca listę.

Wątek korzysta z tego samego obiektu. Proces `spawn` otrzymuje zwykłą listę przez serializację — zmienia własną kopię.
Jawnie zwrócony wynik procesu trzeba odebrać.

Przykład: [memory_demo.py](part05_files/memory_demo.py).

In [ ]:
memory_script = Path("part05_files/memory_demo.py").resolve()
memory_run = subprocess.run(
    [sys.executable, str(memory_script)], check=True,
    capture_output=True, text=True, encoding="utf-8",
    env={**os.environ, "PYTHONIOENCODING": "utf-8"}, timeout=60)
print(memory_run.stdout)

### Rozszerzenie: oddzielna pamięć nie oznacza braku komunikacji
W tym przykładzie nie ma pamięci współdzielonej ani proxy.
Procesy mogą przekazywać wyniki i komunikaty; dostępne są też jawne mechanizmy współdzielenia.

Wątek zmienia listę główną, ale nie zwalnia nas to z synchronizacji bardziej złożonych operacji na wspólnym stanie.

### Rozszerzenie: wymuszony wyścig
Dwa wątki odczytują licznik **przed** zapisem. `Barrier(2)` wymusza dojście obu do tego miejsca.

Każdy zapisuje odczytaną wartość powiększoną o jeden. Ile wyniesie licznik?
To demonstracja złożonej operacji odczyt–zmiana–zapis; nie polegamy na losowym przełączaniu wątków.

In [ ]:
from threading import Barrier, Lock

shared_counter = {"value": 0}
both_read = Barrier(2)

def unsafe_increment():
    previous = shared_counter["value"]
    both_read.wait(timeout=5)
    shared_counter["value"] = previous + 1

with ThreadPoolExecutor(max_workers=2) as executor:
    jobs = [executor.submit(unsafe_increment) for _ in range(2)]
    for job in jobs:
        job.result(timeout=10)
assert shared_counter["value"] == 1
print("Dwa zwiększenia, wynik:", shared_counter["value"])

### Rozszerzenie: blokada obejmuje całą operację
Chronimy wspólnie odczyt i zapis. GIL nie gwarantuje atomowości dowolnej sekwencji instrukcji.

Usuwamy barierę z sekcji krytycznej: oczekiwanie na drugi wątek przy trzymanej blokadzie uniemożliwiłoby mu wejście do tej sekcji.

In [ ]:
shared_counter = {"value": 0}
counter_lock = Lock()

def safe_increment():
    with counter_lock:
        previous = shared_counter["value"]
        shared_counter["value"] = previous + 1

with ThreadPoolExecutor(max_workers=2) as executor:
    jobs = [executor.submit(safe_increment) for _ in range(2)]
    for job in jobs:
        job.result(timeout=10)
assert shared_counter["value"] == 2
print("Z blokadą:", shared_counter["value"])

### Rozszerzenie: gotowy wynik za wolnym zadaniem
Pierwsze zadanie czeka na zdarzenie, drugie od razu zwraca wynik.
`map()` musi zacząć od pierwszego wyniku, choć drugi jest gotowy.

Użyjemy osobnego wątku do odebrania pierwszego elementu iteratora. Główny wątek może sprawdzić stan i zwolnić zadanie bez zakleszczenia.

In [ ]:
first_gate, second_ready = Event(), Event()

def ordered_task(number):
    if number == 0:
        if not first_gate.wait(timeout=5):
            raise TimeoutError("Nie zwolniono pierwszego zadania")
    else:
        second_ready.set()
    return number

with ThreadPoolExecutor(max_workers=2) as workers:
    with ThreadPoolExecutor(max_workers=1) as reader:
        values = workers.map(ordered_task, [0, 1])
        first_value = reader.submit(next, values)
        try:
            assert second_ready.wait(timeout=3)
            assert not first_value.done()
        finally:
            first_gate.set()
        ordered_values = [first_value.result(timeout=3), next(values)]
assert ordered_values == [0, 1]
print("Drugie zadanie było gotowe; map oddał:", ordered_values)

### Rozszerzenie: jak odebrać gotowe wyniki wcześniej?
Wcześniejszy przykład `as_completed()` odbierał **B przed A** dzięki jawnej blokadzie A.

Tu widzimy drugą stronę porównania: `map()` zachowuje kolejność wejścia kosztem oczekiwania na pierwszy wynik.

Użyj `as_completed()`, gdy możesz przetwarzać wyniki bez kolejności. Dołącz identyfikator zadania, jeśli później trzeba ją odtworzyć.

### Rozszerzenie: timeout nie zatrzymuje zadania
`future.result(timeout=...)` ogranicza oczekiwanie wywołującego.
Nie przerywa uruchomionego obliczenia.

`cancel()` działa dla zadania, które jeszcze nie wystartowało. Wyjście z kontekstu executora domyślnie czeka na zadania. Nie traktuj timeoutu jako uniwersalnego mechanizmu zabijania pracy.

### Dlaczego procesy mogą przegrać?
Koszt obejmuje start interpreterów, serializację argumentów, komunikację międzyprocesową i odbiór wyników.

Praca na jednym wierszu pięciu liczb jest bardzo krótka. W takim zadaniu narzut może dominować; najpierw sprawdź wektoryzację.

### `chunksize` i paczki
- `chunksize` grupuje zadania wysyłane przez `map()`; większe paczki ograniczają koszt komunikacji.
- Za duże paczki mogą pogorszyć rozkład nierównej pracy.
- `count_batch()` przetwarza wiele wierszy w jednym wywołaniu.

W `ProcessPoolExecutor.map()` `chunksize` ma znaczenie; w `ThreadPoolExecutor.map()` nie ma efektu.

### Rozszerzenie: pomiar na tym komputerze
Uruchom w terminalu środowiska kursu:
```text
python part05_files/parallel_demo.py --benchmark
```
Skrypt sprawdza wyniki i podaje medianę trzech prób dla pętli, NumPy, dwóch `chunksize` i paczek.
Osobno porównuje większe zadania CPU. Nie zakłada przyspieszenia.

### Jak interpretować porównanie?
Dane, konwersja do listy i referencyjny wynik powstają przed pomiarem.
Czas wariantów procesowych zawiera utworzenie i zamknięcie puli oraz komunikację.

To koszt jednorazowego uruchomienia. W usłudze z długo żyjącą pulą rozkład kosztów będzie inny. Nie przenoś czasów między komputerami.

### Rozszerzenie: ograniczenia przyspieszenia
Jeśli część programu musi pozostać sekwencyjna, dodatkowe rdzenie jej nie przyspieszą.

Dla udziału sekwencyjnego `s` i `p` procesorów idealna granica to:
`speedup <= 1 / (s + (1 - s) / p)`.

Rzeczywisty program ma dodatkowo narzut komunikacji i zarządzania.

### Rozszerzenie: Python 3.14
- `InterpreterPoolExecutor`: osobne interpretery i ich GIL, izolacja stanu oraz serializacja danych.
- Tryb free-threaded jest oficjalnie wspierany, ale pozostaje opcjonalną konfiguracją; nie usuwa potrzeby synchronizacji.
- Na wspieranych POSIX domyślny start procesów zmieniono na `forkserver`; Windows używa `spawn`.

To kontekst rozwoju narzędzi, nie wymagania środowiska 3.10.

### Rozszerzenie: kontrola napływu zadań
W Pythonie 3.14 `Executor.map(..., buffersize=...)` pozwala ograniczyć liczbę zgłoszonych zadań, których wyników jeszcze nie odebrano.

W Pythonie 3.10 tego parametru nie ma. Dla dużych danych projektuj ograniczone paczki zamiast bezwarunkowo tworzyć miliony futures.

## B. Biblioteka standardowa w praktyce
Wybieramy narzędzie do problemu: nazwane rekordy, grupowanie, zliczanie, kolejki, dopasowanie tekstu i losowanie.

Podstawowe `any`, `all`, `sum` oraz iteratory znamy z `part04`; tutaj skupiamy się na nowych zastosowaniach i pułapkach.

In [ ]:
from collections import namedtuple, defaultdict, Counter, deque
from itertools import chain
import math
import re
import random
import secrets

### `namedtuple`: nazwy zamiast indeksów
Niemutowalny rekord zachowuje indeksowanie i rozpakowywanie krotki, a pola mają czytelne nazwy.

Nie oznacza to głębokiej niemutowalności: pole może zawierać mutowalny obiekt.

In [ ]:
Point = namedtuple("Point", ["x", "y"])
p = Point(11, y=22)
other = Point(8, 18)
x, y = p
assert p[0] == p.x == x == 11
assert math.hypot(p.x - other.x, p.y - other.y) == 5
print(p)
print(p._replace(x=0))  # nowy rekord
assert p.x == 11

### Dobierz rodzaj rekordu
- `namedtuple`: mały rekord z interfejsem krotki.
- `typing.NamedTuple`: podobny rekord z adnotacjami typów.
- `dataclass`: klasa danych, domyślnie mutowalna, z większą swobodą rozbudowy.

Nie ma reguły „zawsze używaj namedtuple”. Klasy i dataclasses rozwijamy w `part07`.

In [ ]:
Color = namedtuple("Color", ["hue", "saturation", "lightness"])
pixel = Color(170, 0.1, 0.6)
if pixel.lightness >= 0.5:
    print("Jasny kolor")
# Nazwa pola wyjaśnia znaczenie; samo pixel[2] tego nie robi.

### `defaultdict`: grupowanie danych
Zamiast ręcznie sprawdzać, czy istnieje klucz, przekazujemy fabrykę wartości.

`list` jest funkcją tworzącą nową listę przy każdym brakującym kluczu. Nie przekazujemy gotowej listy `[]`.

In [ ]:
pairs = [("yellow", 1), ("blue", 2), ("yellow", 3)]
grouped = defaultdict(list)
for key, value in pairs:
    grouped[key].append(value)
assert dict(grouped) == {"yellow": [1, 3], "blue": [2]}
assert grouped["yellow"] is not grouped["blue"]
print(dict(grouped))

### Pułapka: odczyt może zmienić słownik
`mapping[key]` uruchamia fabrykę dla nieobecnego klucza i zapisuje wartość.

`.get()` i operator `in` nie uruchamiają fabryki.

In [ ]:
before = set(grouped)
assert grouped.get("red") is None
assert "red" not in grouped
assert set(grouped) == before
assert grouped["red"] == []
assert "red" in grouped
print(dict(grouped))

### `Counter`: od pętli do intencji
`Counter` zlicza obiekty hashowalne. Można go aktualizować kolejnymi porcjami danych.

Brakujący klucz zwraca zero bez tworzenia wpisu. `most_common()` pokazuje najczęstsze elementy.

In [ ]:
counts = Counter("mississippi")
assert counts == {"m": 1, "i": 4, "s": 4, "p": 2}
assert counts["z"] == 0 and "z" not in counts
assert counts.total() == 11  # metoda od Pythona 3.10
print(counts.most_common(2))
words = Counter(["red", "blue", "red"])
words.update(["blue", "green"])
print(words)

### `Counter`: dwa znaczenia odejmowania
Operator `-` usuwa z wyniku liczniki niedodatnie.
Metoda `.subtract()` zmienia obiekt i zachowuje zera oraz liczby ujemne.

To ważna różnica np. między multizbiorem a bilansem zapasów.

In [ ]:
left, right = Counter(a=2, b=1), Counter(a=3, b=1)
assert left - right == Counter()
balance = left.copy()
balance.subtract(right)
assert dict(balance) == {"a": -1, "b": 0}
combined = Counter("which") + Counter("witch")
assert combined == Counter({"w": 2, "h": 3, "i": 2, "c": 2, "t": 1})
print(dict(balance), combined)

### Rozszerzenie: czy wystarczy składników?
`Counter` może opisywać multizbiór składników i wymagania przepisu.

Oblicz braki oraz to, co zostanie po zrealizowaniu zamówienia. Następnie porównaj multizbiór z bilansem zachowującym niedobór jako liczbę ujemną.

In [ ]:
stock = Counter({"jajko": 4, "mleko": 2, "mąka": 1})
recipe = Counter({"jajko": 2, "mleko": 3})
missing = recipe - stock
remaining = stock - recipe
balance = stock.copy()
balance.subtract(recipe)
assert missing == {"mleko": 1}
assert remaining == {"jajko": 2, "mąka": 1}
assert dict(balance) == {"jajko": 2, "mleko": -1, "mąka": 1}
assert stock["mleko"] == 2  # nie zmieniliśmy stanu magazynu
print("Brakuje:", dict(missing))
print("Bilans:", dict(balance))

### `deque`: kolejka z dwóch końców
`append()` i `popleft()` nadają się do kolejki FIFO, bez przesuwania pozostałych elementów jak w `list.pop(0)`.

Opcjonalne `maxlen` ogranicza długość: dopisanie kolejnego elementu usuwa najstarszy z przeciwnego końca.

In [ ]:
queue = deque(["A", "B"])
queue.append("C")
assert queue.popleft() == "A"
history = deque(maxlen=3)
history.extend([1, 2, 3, 4])
assert list(history) == [2, 3, 4]
print(list(queue), list(history))

### `re`: wybierz zakres dopasowania
- `search()` szuka w dowolnym miejscu.
- `match()` sprawdza początek tekstu.
- `fullmatch()` wymaga dopasowania całego tekstu.

Brak dopasowania oznacza `None`. Przed `.group()` trzeba to uwzględnić.

In [ ]:
pattern = re.compile(r"(?P<name>[A-Za-z]+)-(?P<number>[0-9]{3})")
text = "prefix Ada-123 suffix"
assert pattern.search(text).group("name") == "Ada"
assert pattern.match(text) is None
assert pattern.fullmatch("Ada-123") is not None
assert pattern.fullmatch("Ada-123 suffix") is None
match = pattern.fullmatch("brak")
if match is None:
    print("Brak dopasowania")

### Grupy i celowy błąd
Grupa `0` oznacza całe dopasowanie. Grupy numerowane zaczynają się od `1`; można też używać nazw.

W tym wzorcu są dwie grupy. Kolejna komórka **celowo** zgłasza `IndexError`.

In [ ]:
match = pattern.fullmatch("Ada-123")
assert match is not None
assert match.group(0) == "Ada-123"
assert match.group(1) == match.group("name") == "Ada"
assert match.group(2) == "123"
print(match.groupdict())

In [ ]:
match.group(3)

### Raw strings i Unicode
Zapis `r"\w+"` oddziela składnię regexa od interpretacji backslashy przez Pythona.

W wzorcach dla `str`, `\w` i `\d` domyślnie uwzględniają Unicode. Gdy format wymaga cyfr ASCII, użyj `[0-9]` lub świadomie flagi `re.ASCII`.
`\w+` nie jest pełną definicją nazwiska.

In [ ]:
assert re.fullmatch(r"\w+", "Łódź") is not None
assert re.fullmatch(r"\d+", "１２３") is not None
assert re.fullmatch(r"[0-9]+", "１２３") is None
assert re.fullmatch(r"\w+", "Łódź", flags=re.ASCII) is None

### Podstawienie i wielokrotne użycie wzorca
`compile()` pozwala nazwać i wielokrotnie stosować wzorzec; nie jest obietnicą przyspieszenia każdej operacji.

Regex poniżej obsługuje tylko uproszczone przykłady adresów, nie waliduje ogólnego formatu e-mail.

In [ ]:
addresses = "sam@go.com poohbear@bears.com"
changed = re.sub(r"@\w+\.com", "@example.edu", addresses)
assert changed == "sam@example.edu poohbear@example.edu"
print(changed)
print([m.groupdict() for m in pattern.finditer("Ada-123 Jan-456")])

### Rozszerzenie: regex czy operacje na tekście?
Dane mają format `klucz=wartość`, a wartość może zawierać kolejne `=`.

Porównaj `split("=", 1)`, `partition("=")` i regex.
Dla prostego separatora metody napisów często lepiej pokazują intencję; regex pomaga, gdy potrzebne są reguły dopasowania.

In [ ]:
setting = "token=abc=def"
key_split, value_split = setting.split("=", 1)
key_part, separator, value_part = setting.partition("=")
setting_pattern = re.compile(r"(?P<key>[^=]+)=(?P<value>.*)")
setting_match = setting_pattern.fullmatch(setting)
assert setting_match is not None
key_regex, value_regex = setting_match.group("key", "value")
assert (key_split, value_split) == (key_part, value_part)
assert (key_regex, value_regex) == ("token", "abc=def")
assert separator == "="
print(key_part, value_part)

### Rozszerzenie: ustal zachowanie dla błędnych danych
Brak separatora: `split()` daje listę jednoelementową, `partition()` zwraca pusty separator, a nasz regex daje `None`.

Żadne z narzędzi nie definiuje za nas formatu danych. Przed wyborem ustal: pusty klucz, pusta wartość, odstępy, wielowierszowość i obsługa błędów.

In [ ]:
assert "brak".split("=", 1) == ["brak"]
assert "brak".partition("=") == ("brak", "", "")
assert setting_pattern.fullmatch("brak") is None
assert setting_pattern.fullmatch("=wartość") is None
assert setting_pattern.fullmatch("klucz=") is not None

key, separator, value = "=wartość".partition("=")
valid = bool(separator and key)
assert not valid
print("Pusty klucz zaakceptowany?", valid)

### Ćwiczenie: tekst → rekordy → statystyka
Przetwórz uproszczony log: `poziom użytkownik`.

Użyj `fullmatch()`, pomiń niepoprawne wiersze i policz poziomy przez `Counter`. Dodatkowo zbierz użytkowników w `defaultdict(set)`.

Na następnym slajdzie rozwiązanie.

In [ ]:
lines = ["INFO Ada", "ERROR Jan", "INFO Ewa", "???", "ERROR Ada"]
log_pattern = re.compile(r"(?P<level>INFO|ERROR) (?P<user>\w+)")
levels = Counter()
users = defaultdict(set)
for line in lines:
    match = log_pattern.fullmatch(line)
    if match is not None:
        level, user = match.group("level", "user")
        levels[level] += 1
        users[level].add(user)
assert levels == {"INFO": 2, "ERROR": 2}
assert users["ERROR"] == {"Jan", "Ada"}
print(levels)

### Przypomnienie: spłaszczanie list
`sum(parts, [])` wielokrotnie kopiuje rosnący prefiks wyniku; dla wielu małych list koszt może być kwadratowy.

Użyj `chain.from_iterable(parts)` albo comprehension. Szerszy przegląd `itertools`, w tym kombinatoryka i iteratory nieskończone, jest w `part04`.

In [ ]:
parts = [[3, 5], [1, 7], [4]]
flat = list(chain.from_iterable(parts))
assert flat == [x for part in parts for x in part]
assert flat == [3, 5, 1, 7, 4]
print(flat)

### `random`: lokalny stan generatora
`random.Random(seed)` tworzy niezależny generator do symulacji i demonstracji.

Nie zakładaj identycznych wyników wszystkich metod między wersjami Pythona. Zapisuj ziarno, wersję i kolejność wywołań.

In [ ]:
rng_a = random.Random(100)
rng_b = random.Random(100)
rolls = [rng_a.randint(1, 6) for _ in range(6)]
assert rolls == [rng_b.randint(1, 6) for _ in range(6)]
assert all(1 <= roll <= 6 for roll in rolls)
print(rolls)
print(rng_a.random())  # 0 <= x < 1
print(rng_a.randrange(0, 101, 2))  # parzysta, 0..100

### Losowanie i mutowanie
- `choice()`: jeden element.
- `sample()`: bez zwracania, nowa lista.
- `choices()`: ze zwracaniem; można podać wagi.
- `shuffle()`: miesza listę w miejscu i zwraca `None`.

W `uniform(a, b)` koniec przedziału może zależeć od zaokrągleń zmiennoprzecinkowych.

In [ ]:
items = list(range(7))
before = items.copy()
returned = rng_a.shuffle(items)
assert returned is None and sorted(items) == before
sample = rng_a.sample(range(5), k=3)
assert len(sample) == len(set(sample)) == 3
assert len(rng_a.choices(["A", "B"], k=5)) == 5
assert rng_a.choice("abc") in "abc"
print(items, sample)
print(rng_a.normalvariate(mu=0, sigma=3))

### Symulacja a bezpieczeństwo
`random` jest deterministyczny i nie służy do generowania haseł lub tokenów uwierzytelniających.

Do takich zastosowań biblioteka standardowa udostępnia `secrets`. W demonstracji generujemy jedynie przykładowy ciąg, bez używania go jako prawdziwego poświadczenia.

In [ ]:
demo_token = secrets.token_hex(16)  # 16 bajtów -> 32 znaki hex
assert len(demo_token) == 32
assert set(demo_token) <= set("0123456789abcdef")
print("Wygenerowano demonstracyjny token długości", len(demo_token))

### Rozszerzenie: podstawowe funkcje — pułapki
Zamiast katalogu funkcji sprawdźmy znaczenie argumentów i wyniku:
`min(..., key=...)` zwraca element, a drugi argument `sum()` jest wartością początkową.

In [ ]:
assert min(["apple", "banana", "pear"], key=len) == "pear"
assert sum([3, 5, 7], 10) == 25
assert round(123.45, 0) == 123.0
assert round(125, -1) == 120  # remis: do parzystej
assert pow(3, 5, 10) == 3
assert divmod(10, 6) == (1, 4)
assert int("1011", 2) == 11
assert chr(ord("ą")) == "ą"

### Rozszerzenie: liczby dziesiętne
`round()` nie zmienia `float` w dokładną liczbę dziesiętną. Wiele ułamków nie ma dokładnej reprezentacji binarnej.

Jeżeli wymagane są określone reguły dziesiętne, rozważ `decimal.Decimal` tworzone z tekstu.

In [ ]:
from decimal import Decimal, ROUND_HALF_UP
assert round(2.675, 2) == 2.67
rounded = Decimal("2.675").quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
assert rounded == Decimal("2.68")
print(rounded)

### Mapa dalszych narzędzi
| Problem | Moduły |
|---|---|
| Pliki i ścieżki | `pathlib`, `csv`, `json` |
| CLI i diagnostyka | `argparse`, `logging`, `pdb` |
| Czas i liczby | `datetime`, `zoneinfo`, `decimal`, `statistics` |
| Testy i inspekcja | `unittest`, `inspect` |

Nie uczymy się numerów rozdziałów dokumentacji. Szukamy narzędzia do konkretnego zadania.

### Materiały historyczne a bieżący Python
`2to3` i `lib2to3` usunięto w Pythonie 3.13 — nie proponujemy ich jako narzędzi do nowych projektów.

Dokumentacja 3.10 służy tu zgodności z interpreterem kursu. Rozwój narzędzi opisujemy osobno z numerem wersji; nie podnosimy automatycznie zależności projektu.

## Pytania na zakończenie
1. Dlaczego pula procesów może być wolniejsza od pętli?
2. Czy `map_async()` zwraca wyniki w kolejności zakończenia?
3. Kiedy odczyt z `defaultdict` tworzy wpis?
4. Czym różni się `Counter.subtract()` od `-`?
5. Kiedy użyjesz `fullmatch()`, a kiedy `search()`?

### Odpowiedzi
1. Koszt startu, serializacji i komunikacji może dominować.
2. Nie — wynik zachowuje kolejność wejścia.
3. Dla brakującego klucza przez `[]`, nie przez `.get()`.
4. `.subtract()` zmienia obiekt i zachowuje liczniki niedodatnie.
5. Cały tekst: `fullmatch()`; fragment w tekście: `search()`.

### Self reading — procesy i narzędzia
- [multiprocessing — Python 3.10](https://docs.python.org/3.10/library/multiprocessing.html)
- [concurrent.futures — Python 3.10](https://docs.python.org/3.10/library/concurrent.futures.html)
- [NumPy: Generator](https://numpy.org/doc/1.26/reference/random/generator.html)
- [Nowości Pythona 3.14](https://docs.python.org/3.14/whatsnew/3.14.html)
- [Executor i buffersize — Python 3.14](https://docs.python.org/3.14/library/concurrent.futures.html)

### Self reading — biblioteka standardowa
- [collections](https://docs.python.org/3.10/library/collections.html)
- [re](https://docs.python.org/3.10/library/re.html)
- [random](https://docs.python.org/3.10/library/random.html) i [secrets](https://docs.python.org/3.10/library/secrets.html)
- [Usunięcia w Pythonie 3.13](https://docs.python.org/3.13/whatsnew/3.13.html#removed-modules-and-apis)
- [XKCD: Random Number](https://xkcd.com/221/) — ilustracja, nie generator do stosowania

### Archiwalne filmy — materiał opcjonalny
- [Film 1](https://www.youtube.com/watch?v=o9pEzgHorH0)
- [Film 2](https://www.youtube.com/watch?v=JYuE8ZiDPl4)

Dostępność i aktualność filmów pozostają niepotwierdzone w przeglądzie. Podstawowym źródłem jest dokumentacja powyżej.